# 03 · Store → update → gate → promote → rollback

The core loop, by hand (Levels 2–3), using the same library functions the automated levels use:

1. **Store:** human-labeled failures become an evaluation dataset.
2. **Update:** a candidate is registered *without* being promoted. It's either an `AGENTS.md` version or a skill.
3. **Gate:** production vs candidate on the dataset and a holdout set the candidate never saw.
4. **Promote:** move `@production`, and keep a receipt.
5. **Rollback:** undo exactly what the receipt says.

In [ ]:
from sia import notebook as nb

nb.setup(backend="offline", llm=False)   # deterministic: code scorers + rule-based analyst

In [ ]:
# Prerequisite (idempotent): v1 in production and a "week" of labeled production traffic.
from sia import tracking, traffic

nb.ensure_production()
if len(tracking.agent_traces()) < 12:
    ids = traffic.send(("traffic",), n=16, seed=1)        # simulated customers -> production traces
    traffic.label(ids, rate=1.0, seed=1)                 # simulated reviewers label every one
print(len(tracking.agent_traces()), "production traces")

## 1 · Store: failures → dataset

Forty thumbs-down on one question are one test, not forty, so records are deduplicated by question. The holdout
cases are kept out on purpose.

In [ ]:
import pandas as pd
from sia import data

experiment_id = tracking.setup()
dataset = data.get_or_create_dataset(experiment_id)
records = data.dedupe([data.trace_to_record(t, "human_fail") for t in data.labeled_failures()])
existing = {r["question"] for r in dataset.to_df()["inputs"]} if len(dataset.to_df()) else set()
new = [r for r in records if r["inputs"]["question"] not in existing]
if new:
    dataset.merge_records(new)
print(data.growth_report(dataset))
dataset.to_df()[["inputs", "expectations"]].head(5)

## 2 · Update: register a candidate (not promoted)

In [ ]:
import difflib, mlflow
from sia import config
from sia.agent import load_agents_md, register_candidate
from sia.scenario import SCENARIO_DIR

fix = (SCENARIO_DIR / "fixes" / "level2_manual_AGENTS.md").read_text()
candidate = register_candidate(fix, "Lead with the answer; knowledge/ is the source of truth; escalate, don't guess",
                               author="notebook", level=2)
prod = load_agents_md("production")
print(f"candidate v{candidate}; production still v{prod.version}\n")
print("".join(difflib.unified_diff(prod.template.splitlines(True), fix.splitlines(True), "production", "candidate")))

## 3 · Gate

The same rules gate every level (`sia/gate.py`):
* every must-pass scorer is at 100%;
* the candidate beats production on the dataset;
* the candidate is not worse on the holdout.

Each side is an MLflow evaluation run, nested under one gate run.

In [ ]:
from sia import gate

decision = gate.compare(candidate, "production", train=data.rows(dataset), holdout=data.holdout_rows())
print(decision.table())
print("\nGATE:", "PASS" if decision.passed else "FAIL", decision.reasons)

## 4 · Promote, and 5 · roll back

`promote()` moves `@production` (and sets `@previous`), then returns a **receipt**. `rollback(receipt)` restores
exactly what it replaced.

In [ ]:
receipt = gate.promote(candidate, reason=f"notebook gate {decision.run_id}")
print("receipt:", receipt)
print("production ->", load_agents_md("production").version)
print("rollback:", gate.rollback(receipt), "| production ->", load_agents_md("production").version)
receipt = gate.promote(candidate, reason="ship it again")      # leave the fix in production
print("production ->", load_agents_md("production").version)

## Skills: the second update target

A skill (`skills/<name>/SKILL.md`) is a procedure the agent loads only for matching questions. Skills are
versioned in the Prompt Registry too (`acme-skill-<name>`), and an agent version is an **AgentSpec**: one
`AGENTS.md` version plus skill versions.

Here, drift questions arrive about escalations, a policy the instructions don't mention.

In [ ]:
from sia.agent import AgentSpec, pin
from sia import improvers, skills

ids = traffic.send(("drift",), n=6, seed=7)
traffic.label(ids, rate=1.0, seed=7)
report = improvers.failure_report(tracking.agent_traces("tags.topic = 'support_process'"))
print(len(report), "failing drift traces; e.g.", report[0]["question"], "->", report[0]["answer"][:80])

In [ ]:
# The rule-based analyst proposes a skill (an LLM analyst would do the same in Level 3).
prod_skills = {}
proposal = improvers.offline_analyze(report, load_agents_md("production").template, skills=prod_skills)
sk = proposal["skill"]
print(proposal["change_summary"], "\n")
print(skills.skill_md(sk["name"], sk["description"], sk["body"]))

In [ ]:
# File it: OpenClaw's Skill Workshop (pending + security scan) if OpenClaw is set up, plus an MLflow skill version.
p = skills.propose(sk["name"], sk["description"], sk["body"], goal="notebook drift fix",
                   evidence=", ".join(r["trace_id"] for r in report[:3]), author="notebook", level=3)
print(p)
spec = AgentSpec("production", ((p.name, p.version),))
print("candidate spec:", pin(spec).label)

In [ ]:
drift_rows = [data.trace_to_record(t, "drift") for t in tracking.agent_traces("tags.topic = 'support_process'")]
decision = gate.compare(spec, "production", train=data.dedupe(drift_rows), holdout=data.holdout_rows())
print(decision.table()); print("GATE:", "PASS" if decision.passed else "FAIL", decision.reasons)
if decision.passed:
    ok, msg = skills.apply(p)               # Workshop apply first (re-scan + rollback metadata), if OpenClaw is set up
    print("workshop:", msg)
    receipt = gate.promote(spec, reason="notebook: drift skill")
    print("receipt:", receipt, "\nproduction is now", pin("production").label)
else:
    skills.reject(p, "; ".join(decision.reasons))

**Next:** 04 · The loop runs itself. The same steps, unattended.